In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 120
pd.options.display.float_format = lambda x: f"{x:,.6f}"

NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if "/mnt/data" not in sys.path:
    sys.path.insert(0, "/mnt/data")

import hedging_analytics as ha

REG_VAL = 100
RUN_ENGINE = None            # None = auto: load saved workbook if present, otherwise rebuild from raw snapshot data
REBALANCE_EVERY_N_SNAPSHOTS = 1
MAX_SNAPSHOT_GROUPS = None
CURRENCIES = None
EXCLUDE_EXPIRY_CROSSING = True
OUTPUT_FILENAME = "hedging_results.xlsx"
EXPORT_TABLES = False
EXPORT_FIGURES_HTML = False

paths = ha.resolve_default_paths(reg_val=REG_VAL, notebook_dir=NOTEBOOK_DIR, output_filename=OUTPUT_FILENAME)
EXPORT_DIR = paths.project_root / "tables" / "hedging_chapter4"

print(paths)

## 1. Load or run the hedging engine

If `RUN_ENGINE = True`, the notebook first **normalizes the perp timestamps to second precision** and then calls the current backbone. That step matters because the option snapshots are stored at second precision, while the provided perp file may carry microseconds.

If `RUN_ENGINE = False`, the notebook loads an existing saved hedging workbook.


In [ ]:
if RUN_ENGINE is None:
    RUN_ENGINE = not paths.output_xlsx.exists()

if RUN_ENGINE:
    results = ha.run_engine_with_normalized_perp(
        hedging_engine_py=paths.hedging_engine_py,
        calibration_xlsx=paths.calibration_xlsx,
        data_dir=paths.data_dir,
        output_xlsx=paths.output_xlsx,
        perp_history_csv=paths.perp_history_csv,
        rebalance_every_n_snapshots=REBALANCE_EVERY_N_SNAPSHOTS,
        max_snapshot_groups=MAX_SNAPSHOT_GROUPS,
        currencies=CURRENCIES,
        exclude_expiry_crossing=EXCLUDE_EXPIRY_CROSSING,
    )
else:
    if not paths.output_xlsx.exists():
        raise FileNotFoundError(
            f"No saved hedging workbook at {paths.output_xlsx}. Either place one there or set RUN_ENGINE = True."
        )
    results = ha.load_output_workbook(paths.output_xlsx)

interval_panel = ha.ensure_analysis_columns(results["hedge_interval_panel"])
interval_long = ha.make_interval_long(interval_panel)

headline_all = ha.pooled_summary(interval_long, ["split", "currency", "model", "hedge_type"])
headline_test = (
    headline_all.loc[headline_all["split"].astype(str).str.lower() == "test"].copy()
    if "split" in headline_all.columns
    else ha.empty_pooled_summary(["split", "currency", "model", "hedge_type"])
)

rep_model = ha.choose_representative_model(headline_test)
rep_test = ha.representative_panel(interval_long, model=rep_model, split="test")
rep_test_series = ha.unhedged_vs_hedged_panel(rep_test)

print(f"RUN_ENGINE = {RUN_ENGINE}")
print(f"Representative model for the deep-dive figures: {rep_model}")
print(f"Interval rows: {len(interval_panel):,}")
print(f"Eligible interval-model-hedge rows: {len(interval_long.loc[interval_long['is_summary_eligible'].fillna(False)]):,}")

## 2. Sample construction and validation

This is the right place to show the reader what actually survives into the interval hedging sample.


In [ ]:
sample_table = ha.build_sample_construction(interval_panel)
display(
    ha.display_ready(
        sample_table,
        percent_cols=[
            "eligible_share",
            "missing_next_option_share",
            "missing_perp_t_share",
            "missing_perp_t1_share",
            "expiry_crossing_share",
            "cheap_option_share",
        ],
        round_digits=4,
    )
)

## 3. Headline pooled performance

These are the core test-sample comparisons. They should answer two questions fast:
1. Does hedging materially reduce interval risk relative to leaving the short option unhedged?
2. Does the premium-adjusted **net delta** matter more than the choice between Black, Heston, and SVCJ?


In [ ]:
headline_cols = [
    "split", "currency", "model", "hedge_type", "n_intervals",
    "rmse_reduction", "mae_reduction", "variance_reduction",
    "q05_hedged_coin", "es05_hedged_coin", "hit_rate_abs_improvement",
    "median_basis_abs", "median_dt_hours",
]

display(
    ha.display_ready(
        headline_test[headline_cols],
        percent_cols=[
            "rmse_reduction",
            "mae_reduction",
            "variance_reduction",
            "hit_rate_abs_improvement",
            "median_basis_abs",
        ],
        round_digits=4,
    )
)

fig_pooled_rmse = ha.figure_pooled_metric(headline_test, metric="rmse_reduction")
fig_pooled_mae = ha.figure_pooled_metric(headline_test, metric="mae_reduction")
fig_pooled_rmse.show()
fig_pooled_mae.show()

## 4. Snapshot-level time-series diagnostics

This figure is useful when you want Chapter 4 to visually match the calibration chapter rather than looking like a single pooled table.


In [ ]:
summary_by_ts = ha.pooled_summary(rep_test, ["snapshot_ts", "currency", "hedge_type", "model"])
fig_snapshot_rmse = ha.figure_snapshot_metric(
    summary_by_ts,
    metric="rmse_reduction",
    representative_model=rep_model,
    hedge_type="net_delta",
)
fig_snapshot_rmse.show()

## 5. Distribution and tail-risk figures

These figures show whether the hedge improves the full distribution of one-step outcomes, not just the average error.


In [ ]:
fig_ecdf = ha.figure_ecdf_abs_pnl(rep_test_series)
fig_tail = ha.figure_tail_comparison(rep_test_series)
fig_ecdf.show()
fig_tail.show()

## 6. Maturity and basis diagnostics

The maturity profile is where basis risk should show up most clearly, because the option is written on a term future but hedged with the perpetual.


In [ ]:
maturity_test = ha.pooled_summary(rep_test, ["currency", "maturity_bucket", "hedge_type", "model"])
fig_maturity_rmse = ha.figure_maturity_metric(
    maturity_test,
    metric="rmse_reduction",
    representative_model=rep_model,
    split="test",
)
fig_basis_maturity = ha.figure_basis_by_maturity(rep_test, split="test")

fig_maturity_rmse.show()
fig_basis_maturity.show()

display(
    ha.display_ready(
        maturity_test[["currency", "maturity_bucket", "hedge_type", "n_intervals", "rmse_reduction", "mae_reduction", "variance_reduction", "median_basis_abs"]],
        percent_cols=["rmse_reduction", "mae_reduction", "variance_reduction", "median_basis_abs"],
        round_digits=4,
    )
)

## 7. Moneyness and basis-quintile diagnostics

The fixed OTM / ATM / ITM buckets are still useful, but for the main text the equal-count moneyness and basis bins are often cleaner.


In [ ]:
fig_moneyness_quintile = ha.figure_moneyness_quintile_metric(rep_test, metric="rmse_reduction", q=5)
fig_basis_quintile = ha.figure_basis_quintile_metric(rep_test, metric="rmse_reduction", q=5)
fig_moneyness_quintile.show()
fig_basis_quintile.show()

## 8. Sensitivity checks

These rows are useful for the thesis text because they show whether the main ranking survives after removing the regions most likely to distort pooled metrics.


In [ ]:
sensitivity_table = ha.build_sensitivity_table(rep_test)
display(
    ha.display_ready(
        sensitivity_table,
        percent_cols=[
            "rmse_reduction",
            "mae_reduction",
            "variance_reduction",
            "hit_rate_abs_improvement",
        ],
        round_digits=4,
    )
)

## 9. Thesis takeaway block

This is a compact text block you can paste into Chapter 4 and then tighten manually once you decide which figures make the final cut.


In [ ]:
if headline_test.empty:
    display(Markdown("No test-sample headline results available."))
else:
    pivot_rmse = headline_test.pivot_table(index=["currency", "model"], columns="hedge_type", values="rmse_reduction")
    rmse_lines = []
    for (currency, model), row in pivot_rmse.sort_index().iterrows():
        delta_val = row.get("delta")
        net_val = row.get("net_delta")
        delta_txt = f"{100 * delta_val:.1f}%" if pd.notna(delta_val) else "n/a"
        net_txt = f"{100 * net_val:.1f}%" if pd.notna(net_val) else "n/a"
        rmse_lines.append(f"- **{currency} / {model.title()}**: delta {delta_txt}, net delta {net_txt}")

    takeaway = f"""
### Draft takeaway

On the **test sample**, the hedging results are dominated by the choice of hedge rule rather than the choice of stochastic model.
Across BTC and ETH, the premium-adjusted **net-delta hedge** consistently delivers materially larger reductions in interval RMSE and MAE than the naive regular-delta hedge.
By contrast, the differences across **Black, Heston, and SVCJ** are economically small once the hedge rule is fixed.

Representative pooled RMSE reductions:
{chr(10).join(rmse_lines)}

The maturity and basis diagnostics suggest that the long end of the surface is where residual hedging frictions become most visible, which is consistent with growing **perpetual–term basis risk**.
"""
    display(Markdown(takeaway))

## 10. Optional export

Turn on `EXPORT_TABLES` and/or `EXPORT_FIGURES_HTML` in the configuration cell to write out thesis-ready artifacts.


In [ ]:
if EXPORT_TABLES or EXPORT_FIGURES_HTML:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

figures = {
    "fig_pooled_rmse": fig_pooled_rmse,
    "fig_pooled_mae": fig_pooled_mae,
    "fig_snapshot_rmse": fig_snapshot_rmse,
    "fig_ecdf": fig_ecdf,
    "fig_tail": fig_tail,
    "fig_maturity_rmse": fig_maturity_rmse,
    "fig_basis_maturity": fig_basis_maturity,
    "fig_moneyness_quintile": fig_moneyness_quintile,
    "fig_basis_quintile": fig_basis_quintile,
}

tables = {
    "sample_construction": sample_table,
    "headline_test": headline_test,
    "maturity_test": maturity_test,
    "sensitivity_table": sensitivity_table,
}

if EXPORT_TABLES:
    ha.export_tables_csv(tables, EXPORT_DIR)
    print("Exported CSV tables to:", EXPORT_DIR)

if EXPORT_FIGURES_HTML:
    ha.export_figures_html(figures, EXPORT_DIR)
    print("Exported HTML figures to:", EXPORT_DIR)